# Neural networks from Chapter 4 in Python

Python/NumPy implementations of `FNN_matlab_code`, `FNN_multiclass_linesearch_code`, and `Chapter.Gradient.Neural`. The module keeps the MATLAB convention that examples are columns and weight matrices contain a bias column.

In [ ]:
import matplotlib.pyplot as plt

"""Implementações NumPy dos três exemplos de redes neurais originalmente em MATLAB."""

import numpy as np


def sigmoid(z):
    """Calcula a função sigmoide de forma numericamente estável.

    Parâmetros
    ----------
    z : array_like
        Valores de entrada da função.

    Retorna
    -------
    numpy.ndarray
        Valores no intervalo [0, 1].
    """
    z = np.asarray(z, dtype=float)
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1.0 + exp_z)
    return result


def activation(z, type):
    """Calcula uma função de ativação e sua derivada.

    Parâmetros
    ----------
    z : array_like
        Pré-ativação da camada.
    type : int
        Tipo da ativação: 1 para sigmoide, 2 para ReLU e 3 para softmax.

    Retorna
    -------
    tuple
        A ativação calculada e sua derivada. Para softmax, a derivada é
        retornada como uma matriz Jacobiana para cada exemplo.
    """
    z = np.asarray(z, dtype=float)
    if type == 1:
        value = sigmoid(z)
        return value, value * (1.0 - value)
    if type == 2:
        return np.maximum(z, 0.0), (z > 0.0).astype(float)
    if type == 3:
        shifted = z - np.max(z, axis=0, keepdims=True)
        exp_z = np.exp(shifted)
        value = exp_z / np.sum(exp_z, axis=0, keepdims=True)
        jacobian = np.empty((z.shape[1], z.shape[0], z.shape[0]))
        for example, probability in enumerate(value.T):
            jacobian[example] = np.diag(probability) - np.outer(probability, probability)
        return value, jacobian
    raise ValueError("type must be 1 (sigmoid), 2 (ReLU), or 3 (softmax)")


def misfit(y_pred, y_obs, scale=1.0, type=1):
    """Calcula a função objetivo e sua derivada em relação à previsão.

    Parâmetros
    ----------
    y_pred : array_like
        Valores previstos pela rede.
    y_obs : array_like
        Valores observados ou rótulos esperados.
    scale : float, default=1.0
        Fator de escala aplicado ao objetivo e à derivada.
    type : int, default=1
        Tipo do objetivo: 1 para erro quadrático L2 e 2 para entropia cruzada.

    Retorna
    -------
    tuple
        Valor do objetivo e gradiente em relação a ``y_pred``.
    """
    y_pred = np.asarray(y_pred, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float)
    if type == 1:
        value = np.sum((y_pred - y_obs) ** 2)
        derivative = 2.0 * (y_pred - y_obs)
    elif type == 2:
        clipped = np.clip(y_pred, 1e-12, 1.0 - 1e-12)
        value = -np.sum(y_obs * np.log(clipped) + (1.0 - y_obs) * np.log(1.0 - clipped))
        derivative = -y_obs / clipped + (1.0 - y_obs) / (1.0 - clipped)
    else:
        raise ValueError("type must be 1 (L2) or 2 (cross-entropy)")
    return scale * value, scale * derivative


def initialize_weights(layer_size, seed=7, scale=0.1):
    """Inicializa aleatoriamente os pesos de todas as camadas.

    A primeira coluna de cada matriz é reservada para os pesos do bias.

    Parâmetros
    ----------
    layer_size : tuple[int, ...]
        Número de neurônios em cada camada, incluindo entrada e saída.
    seed : int, default=7
        Semente do gerador aleatório.
    scale : float, default=0.1
        Fator multiplicativo dos pesos iniciais.

    Retorna
    -------
    list[numpy.ndarray]
        Matrizes de pesos com formato
        ``(neurônios_da_camada_atual, neurônios_da_camada_anterior + 1)``.
    """
    generator = np.random.default_rng(seed)
    return [
        generator.random((layer_size[i + 1], layer_size[i] + 1)) * scale
        for i in range(len(layer_size) - 1)
    ]


def _forward(x, weights, hidden_type=1, output_type=1):
    """Executa a propagação direta da rede.

    A função adiciona uma linha de bias antes de cada multiplicação matricial.
    Ela é interna ao módulo e retorna tanto as ativações quanto as
    pré-ativações, necessárias no cálculo da retropropagação.
    """
    activations = [np.asarray(x, dtype=float)]
    preactivations = []
    for index, weight in enumerate(weights):
        input_with_bias = np.vstack((np.ones((1, activations[-1].shape[1])), activations[-1]))
        z = weight @ input_with_bias
        value, _ = activation(z, output_type if index == len(weights) - 1 else hidden_type)
        preactivations.append(z)
        activations.append(value)
    return activations, preactivations


def _validate_network(x, weights, layer_size):
    """Valida as dimensões da entrada, dos pesos e da arquitetura."""
    if len(weights) != len(layer_size) - 1 or x.shape[0] != layer_size[0]:
        raise ValueError("incompatible network dimensions")
    for index, weight in enumerate(weights):
        expected = (layer_size[index + 1], layer_size[index] + 1)
        if weight.shape != expected:
            raise ValueError(f"weight {index} has shape {weight.shape}; expected {expected}")


def gradientnn(M, x, t, ww, layer_size, obj_option=1, act_option=1, lambda_=0.0):
    """Calcula o objetivo e os gradientes de uma FNN binária.

    Esta função corresponde a ``FNN_matlab_code/gradientnn.m``. A saída usa
    a mesma ordem conceitual do código MATLAB: lista de gradientes e valor do
    objetivo.

    Parâmetros
    ----------
    M : int
        Número de exemplos de treinamento.
    x : numpy.ndarray
        Dados de entrada com formato ``(características, exemplos)``.
    t : numpy.ndarray
        Rótulos binários.
    ww : list[numpy.ndarray]
        Matrizes de pesos da rede.
    layer_size : tuple[int, ...]
        Arquitetura incluindo camada de entrada e camada de saída.
    obj_option : int, default=1
        1 para L2 ou 2 para entropia cruzada.
    act_option : int, default=1
        1 para sigmoide ou 2 para ReLU nas camadas ocultas.
    lambda_ : float, default=0.0
        Coeficiente da regularização L2.

    Retorna
    -------
    tuple
        ``(gradientes, objetivo)``.
    """
    x = np.asarray(x, dtype=float)
    t = np.asarray(t, dtype=float).reshape(1, -1)
    if x.shape[1] != M or t.shape[1] != M:
        raise ValueError("M must equal the number of examples")
    _validate_network(x, ww, layer_size)
    activations, preactivations = _forward(x, ww, act_option, 1)
    objective, signal = misfit(activations[-1], t, 1.0 / M, obj_option)
    objective += lambda_ / (2.0 * M) * sum(np.sum(weight ** 2) for weight in ww)
    gradients = [None] * len(ww)
    for index in range(len(ww) - 1, -1, -1):
        derivative = activation(preactivations[index], 1 if index == len(ww) - 1 else act_option)[1]
        delta = signal * derivative
        previous = np.vstack((np.ones((1, M)), activations[index]))
        gradients[index] = delta @ previous.T + (lambda_ / M) * ww[index]
        signal = (ww[index].T @ delta)[1:, :]
    return gradients, float(objective)


def make_binary_data(M=100, N=5, seed=7):
    """Gera dados binários sintéticos semelhantes ao exemplo MATLAB.

    Aproximadamente metade dos exemplos recebe o vetor de entrada nulo e o
    rótulo 1; os demais exemplos são vetores binários com rótulo 0.
    """
    generator = np.random.default_rng(seed)
    x = generator.integers(0, 2, size=(N, M)).astype(float)
    t = (np.sum(x, axis=0) == 0).astype(float)
    candidates = np.flatnonzero(t == 0)
    extra = max(0, M // 2 - int(np.sum(t)))
    if extra:
        x[:, candidates[:extra]] = 0.0
        t[candidates[:extra]] = 1.0
    order = generator.permutation(M)
    return x[:, order], t[order].reshape(1, -1)


def train_binary_line_search(
    x, t, layer_size=(5, 10, 1), obj_option=1, act_option=1,
    iterations=400, lambda_=0.0, limit=1e-5, seed=7
):
    """Treina uma FNN binária com descida do gradiente e busca linear.

    Em cada iteração, calcula o gradiente e reduz o passo pela metade enquanto
    a atualização não diminuir o valor da função objetivo.
    """
    weights = initialize_weights(layer_size, seed)
    history = []
    for _ in range(iterations):
        gradients, value = gradientnn(x.shape[1], x, t, weights, layer_size,
                                      obj_option, act_option, lambda_)
        step = 1.0
        candidate = [weight - step * gradient for weight, gradient in zip(weights, gradients)]
        candidate_value = gradientnn(x.shape[1], x, t, candidate, layer_size,
                                     obj_option, act_option, lambda_)[1]
        while candidate_value > value and step > limit:
            step *= 0.5
            candidate = [weight - step * gradient for weight, gradient in zip(weights, gradients)]
            candidate_value = gradientnn(x.shape[1], x, t, candidate, layer_size,
                                         obj_option, act_option, lambda_)[1]
        weights = candidate
        history.append(value)
    return weights, np.asarray(history)


def flatten_weights(weights):
    """Concatena todas as matrizes de pesos em um vetor unidimensional."""
    return np.concatenate([weight.ravel() for weight in weights])


def unflatten_weights(parameters, layer_size):
    """Reconstrói as matrizes de pesos a partir de um vetor de parâmetros."""
    parameters = np.asarray(parameters, dtype=float).ravel()
    weights = []
    position = 0
    for index in range(len(layer_size) - 1):
        shape = (layer_size[index + 1], layer_size[index] + 1)
        size = shape[0] * shape[1]
        weights.append(parameters[position:position + size].reshape(shape))
        position += size
    if position != parameters.size:
        raise ValueError("parameter vector has an invalid length")
    return weights


def gradientnn_fmincg(nn_params, M, x, t, layer_size, obj_option=2,
                      act_option=1, lambda_=0.0):
    """Calcula objetivo e gradiente da FNN multiclasses com softmax.

    Corresponde a ``FNN_multiclass_linesearch_code/gradientnn_fmincg.m``.
    Os pesos são recebidos em um único vetor para uso pelo otimizador de
    gradiente conjugado.
    """
    weights = unflatten_weights(nn_params, layer_size)
    x = np.asarray(x, dtype=float)
    t = np.asarray(t, dtype=float)
    _validate_network(x, weights, layer_size)
    activations, preactivations = _forward(x, weights, act_option, 3)
    objective, signal = misfit(activations[-1], t, 1.0 / M, obj_option)
    objective += lambda_ / (2.0 * M) * sum(np.sum(weight ** 2) for weight in weights)
    gradients = [None] * len(weights)
    for index in range(len(weights) - 1, -1, -1):
        output_layer = index == len(weights) - 1
        derivative = activation(preactivations[index], 3 if output_layer else act_option)[1]
        delta = np.einsum("eij,je->ie", derivative, signal) if output_layer else signal * derivative
        previous = np.vstack((np.ones((1, M)), activations[index]))
        gradients[index] = delta @ previous.T + (lambda_ / M) * weights[index]
        signal = (weights[index].T @ delta)[1:, :]
    return float(objective), flatten_weights(gradients)


def conjugate_gradient_line_search(function, initial, iterations=100):
    """Otimiza uma função pelo método de gradiente conjugado de Polak-Ribiere.

    A busca linear usa condições de Wolfe simplificadas e limita o número de
    tentativas de redução do passo para evitar atualizações instáveis.
    """
    parameters = np.asarray(initial, dtype=float).copy()
    value, gradient = function(parameters)
    direction = -gradient
    history = [value]
    for _ in range(iterations):
        slope = float(gradient @ direction)
        if slope >= 0.0:
            direction = -gradient
            slope = -float(gradient @ gradient)
        step = 1.0
        for _ in range(30):
            candidate = parameters + step * direction
            candidate_value, candidate_gradient = function(candidate)
            candidate_slope = float(candidate_gradient @ direction)
            if candidate_value <= value + 0.01 * step * slope and abs(candidate_slope) <= 0.5 * abs(slope):
                break
            step *= 0.5
        else:
            break
        parameters, value = candidate, candidate_value
        old_gradient, gradient = gradient, candidate_gradient
        beta = max(0.0, float(gradient @ (gradient - old_gradient)) /
                   max(float(old_gradient @ old_gradient), np.finfo(float).eps))
        direction = -gradient + beta * direction
        history.append(value)
    return parameters, np.asarray(history)


def make_multiclass_data(classes=6, examples_per_class=50, seed=7):
    """Gera dados sintéticos para classificação multiclasses.

    Cada classe é representada por um padrão binário permutado aleatoriamente.
    Os rótulos são retornados no formato one-hot usado pela camada softmax.
    """
    generator = np.random.default_rng(seed)
    features = classes - 1
    x = np.zeros((features, classes * examples_per_class))
    t = np.zeros((classes, classes * examples_per_class))
    position = 0
    for class_index in range(classes):
        template = np.r_[np.ones(class_index), np.zeros(features - class_index)]
        for _ in range(examples_per_class):
            x[:, position] = template[generator.permutation(features)]
            t[class_index, position] = 1.0
            position += 1
    order = generator.permutation(x.shape[1])
    return x[:, order], t[:, order]


def train_multiclass(x, t, hidden=(10,), iterations=100, act_option=1,
                     lambda_=0.0, seed=7):
    """Treina uma FNN multiclasses com saída softmax.

    Parâmetros
    ----------
    x : numpy.ndarray
        Entrada com exemplos organizados nas colunas.
    t : numpy.ndarray
        Rótulos one-hot, também com exemplos nas colunas.
    hidden : tuple[int, ...], default=(10,)
        Número de neurônios nas camadas ocultas.
    iterations : int, default=100
        Número máximo de buscas lineares.
    act_option : int, default=1
        Ativação das camadas ocultas: 1 para sigmoide ou 2 para ReLU.
    lambda_ : float, default=0.0
        Coeficiente da regularização L2.
    seed : int, default=7
        Semente da inicialização dos pesos.

    Retorna
    -------
    tuple
        Pesos treinados, histórico do objetivo e descrição da arquitetura.
    """
    layer_size = (x.shape[0], *hidden, t.shape[0])
    initial = flatten_weights(initialize_weights(layer_size, seed))
    objective = lambda parameters: gradientnn_fmincg(
        parameters, x.shape[1], x, t, layer_size, 2, act_option, lambda_)
    parameters, history = conjugate_gradient_line_search(objective, initial, iterations)
    return unflatten_weights(parameters, layer_size), history, layer_size


def numerical_gradients(x, t, weights, layer_size, obj_option, act_option, epsilon=1e-5):
    """Calcula os gradientes por diferenças finitas centrais."""
    numeric = [np.zeros_like(weight) for weight in weights]
    for layer, weight in enumerate(weights):
        for index in np.ndindex(weight.shape):
            plus = [item.copy() for item in weights]
            minus = [item.copy() for item in weights]
            plus[layer][index] += epsilon
            minus[layer][index] -= epsilon
            plus_value = gradientnn(x.shape[1], x, t, plus, layer_size,
                                    obj_option, act_option)[1]
            minus_value = gradientnn(x.shape[1], x, t, minus, layer_size,
                                     obj_option, act_option)[1]
            numeric[layer][index] = (plus_value - minus_value) / (2.0 * epsilon)
    return numeric


def gradient_check(x, t, layer_size=(5, 10, 1), epsilon=1e-5, seed=7):
    """Compara gradientes analíticos e numéricos.

    A comparação é executada para as combinações de objetivo L2/entropia
    cruzada e ativação sigmoide/ReLU. O resultado é um dicionário com o erro
    relativo de cada combinação.
    """
    weights = initialize_weights(layer_size, seed)
    reports = {}
    for objective in (1, 2):
        for activation_type in (1, 2):
            analytical = gradientnn(x.shape[1], x, t, weights, layer_size,
                                     objective, activation_type)[0]
            numerical = numerical_gradients(x, t, weights, layer_size,
                                            objective, activation_type, epsilon)
            difference = np.linalg.norm(flatten_weights(analytical) - flatten_weights(numerical))
            scale = np.linalg.norm(flatten_weights(analytical)) + np.linalg.norm(flatten_weights(numerical))
            reports[(objective, activation_type)] = difference / max(scale, 1e-15)
    return reports


def predict_binary(x, weights, act_option=1):
    """Retorna as probabilidades previstas pela FNN binária."""
    return _forward(x, weights, act_option, 1)[0][-1]


def predict_multiclass(x, weights, act_option=1):
    """Retorna o índice da classe mais provável para cada exemplo."""
    return np.argmax(_forward(x, weights, act_option, 3)[0][-1], axis=0)

np.set_printoptions(precision=4, suppress=True)


## `FNN_matlab_code`: binary FNN with bisection line search

In [ ]:
x_binary, t_binary = make_binary_data(M=100, N=5, seed=7)
binary_weights, binary_history = train_binary_line_search(
    x_binary, t_binary, layer_size=(5, 10, 1), obj_option=1,
    act_option=1, iterations=400, lambda_=0.01, seed=7
)
binary_prediction = predict_binary(x_binary, binary_weights)
print(f'Final binary objective: {binary_history[-1]:.6f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(binary_history)
axes[0].set(title='Binary convergence', xlabel='Iteration', ylabel='Objective')
axes[1].plot(t_binary.ravel(), 'r*', label='Observed')
axes[1].plot(binary_prediction.ravel(), 'g.', label='Predicted probability')
axes[1].set(title='Binary predictions', xlabel='Example', ylabel='Class/value')
axes[1].legend()
plt.tight_layout()
plt.show()

## `FNN_multiclass_linesearch_code`: softmax FNN with conjugate-gradient line search

In [ ]:
x_multi, t_multi = make_multiclass_data(classes=6, examples_per_class=50, seed=7)
multi_weights, multi_history, multi_layer_size = train_multiclass(
    x_multi, t_multi, hidden=(10,), iterations=80, act_option=1, seed=7
)
observed = np.argmax(t_multi, axis=0)
predicted = predict_multiclass(x_multi, multi_weights)
print(f'Multiclass accuracy: {np.mean(predicted == observed):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(multi_history)
axes[0].set(title='Multiclass convergence', xlabel='Line search', ylabel='Objective')
axes[1].plot(observed, 'r*')
axes[1].set(title='Observed classes', xlabel='Example', ylabel='Class')
axes[2].plot(predicted, 'g*')
axes[2].set(title='Predicted classes', xlabel='Example', ylabel='Class')
plt.tight_layout()
plt.show()

## `Chapter.Gradient.Neural`: numerical gradient check

In [ ]:
gradient_errors = gradient_check(x_binary, t_binary, layer_size=(5, 10, 1), seed=7)
for (objective, activation), error in gradient_errors.items():
    print(f'objective={objective}, activation={activation}: relative error={error:.3e}')

The implementation uses stable sigmoid/softmax calculations, validates network dimensions, and handles the softmax/cross-entropy derivative as `probabilities - one_hot_targets`. This is the vectorized form of the Jacobian multiplication used in the MATLAB code.